# k-means clustering

_Machine Learning_

**Group unlabeled points into k clusters around moving centers.**

Sometimes data has no labels. We want to find natural groups anyway. That is clustering.

     k-means finds $k$ groups, each with a center called a centroid.

---

This notebook builds k-means up one step at a time: first see the result on clean blobs, then watch it rediscover real wine cultivars from chemistry alone. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We start on synthetic **blobs** — three well-separated clouds — so the algorithm has an easy target. We do this in two steps: (1) fit k-means with the right number of clusters and inspect what it found, then (2) sweep `k` to see how cluster tightness responds.

### Step 1 — Make blobs and fit k-means

`make_blobs` draws 400 points from 3 Gaussian clusters, so we know the ground-truth grouping. We fit `KMeans` with `n_clusters=3`. The `cluster_centers_` are the learned centroids, and `inertia_` is the total squared distance from each point to its assigned centroid — lower means tighter clusters.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

# 400 points drawn from 3 well-separated Gaussian clusters.
X, true_labels = make_blobs(n_samples=400, centers=3, cluster_std=0.8,
                            random_state=0)

# Fit k-means with the matching number of clusters.
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)

print("cluster centers:")
print(np.round(km.cluster_centers_, 2))
print("inertia (tightness):", round(km.inertia_, 1))

### Step 2 — Sweep k and watch inertia drop

Inertia always falls as `k` grows — more centroids can always sit closer to the points. The interesting question is *where the drop slows down*. That bend (the "elbow") hints at the natural number of clusters; here it appears at `k=3`, matching how we built the data.

In [ ]:
# The elbow: inertia keeps dropping as k grows, but the gains shrink.
for k in [1, 2, 3, 4]:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    print("k=%d  inertia=%.1f" % (k, km_k.inertia_))

## Visualize the data & results

_The wine dataset has 178 wines from 3 cultivars. Can k-means rediscover the groups from the chemistry alone?_

### Step 1 — Load real wine chemistry and reduce to 2D

The wine dataset has 178 wines with 13 chemical measurements each. We **standardise** the features (each on a different scale) so distances are comparable, then use **PCA** to project down to 2 components purely so we can draw the result on a flat plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 178 real wines, 13 chemical measurements each.
wine = load_wine()

# Standardise so every measurement is on a comparable scale.
X = StandardScaler().fit_transform(wine.data)

# Project to 2D so we can plot the clusters.
P = PCA(n_components=2, random_state=0).fit_transform(X)

# Cluster the 2D points into 3 groups.
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(P)

### Step 2 — Plot the clusters and the elbow side by side

Left: each point coloured by its assigned cluster, with the orange X markers showing the learned centroids — see whether k-means recovered the three cultivars from chemistry alone. Right: the elbow plot over `k = 1..5`, where the bend again suggests 3 is the natural number of groups.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left — each point coloured by its k-means cluster.
colors = ["#4ea1ff", "#7ee787", "#c89bff"]
for c in range(3):
    pts = P[km.labels_ == c]
    ax1.scatter(pts[:, 0], pts[:, 1], c=colors[c], edgecolor="k")

# Overlay the learned centroids.
cen = km.cluster_centers_
ax1.scatter(cen[:, 0], cen[:, 1], c="#ffb454", marker="X", s=200,
            edgecolor="k", label="centroids")
ax1.set_xlabel("PCA component 1")
ax1.set_ylabel("PCA component 2")
ax1.set_title("k-means on wine chemistry")
ax1.legend()

# Right — elbow plot of inertia vs k.
ks = [1, 2, 3, 4, 5]
inertia = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(P).inertia_ for k in ks]
ax2.plot(ks, inertia, color="#ffb454", marker="o", label="inertia")
ax2.set_xlabel("k (clusters)")
ax2.set_ylabel("inertia")
ax2.set_title("Elbow plot")
ax2.legend()

plt.show()